<a href="https://colab.research.google.com/github/Ecc535/Mental_Health_LLM_Benchmarking/blob/main/Flan_T5_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers accelerate torch

In [2]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

In [ ]:
model_id = "google/flan-t5-xl"
print(f"Loading {model_id}...")

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl", device_map="auto")

In [4]:
FEATURE_GROUPS = {
    "Rhythm": [
        "steps_48phi", "steps_8phi", "steps_psd_auc_64h",
        "steps_12p_reject", "steps_128p_reject", "steps_IV"
    ],
    "Heart Rate Variability": [
        "hrstd_max", "hrstd_std", "hrmax_std", "hrmean_max",
        "hrmax_mean", "hrentropy_std", "RestingHeartRate_std", "hrmax_max"
    ],
    "Shift Pattern": [
        "shift_hours_median", "shift_type_shannon_entropy",
        "shift_hours_shannon_entropy"
    ],
    "Inactivity at Daytime": [
        "duration_entropy_non_step_std"
    ],
    "Morning Energy": [
        "happiness_morning_std", "happiness_morning_shannon_entropy",
        "sleepiness_daytime_shannon_entropy", "energy_morning_std",
        "alterness_morning_std", "happiness_morning_median"
    ],
    "Evening Energy": [
        "energy_evening_median", "relax_evening_median",
        "health_evening_std", "energy_evening_std",
        "health_evening_median", "happiness_evening_median",
        "happiness_evening_std"
    ],
    "Caffeine Intake": [
        "caffeine_cups_shannon_entropy",
        "awake_type_shannon_entropy",
        "awake_type_std",
        "awake_type_median"
    ],
    "Sleep": [
        "sleep_efficiency_median",
        "sleep_duration_max",
        "nap_duration_fitbit_max"
    ]
}

In [5]:
def build_prompt(grouped_dict):
    formatted_sections = []

    for group, features in grouped_dict.items():
        section = f"## {group}\n"
        for k, v in features.items():
            section += f"{k}: {v}\n"
        formatted_sections.append(section)

    data_block = "\n".join(formatted_sections)

    prompt = f"""
Data:
{data_block}

Write a concise professional summary describing:
- Circadian rhythm stability
- Cardiovascular variability
- Work/shift regularity
- Daytime inactivity
- Morning and evening energy patterns
- Sleep quality

Interpret patterns at a high level.
Do NOT list features individually.
Do NOT invent values.

Summary:
"""
    return prompt.strip()

In [6]:
def build_classification_prompt(summary):
    return f"""Read the following behavioral health summary and classify the burnout risk.

Summary:
{summary}

Question: Based on the summary, is the burnout risk level High or Low?
Answer:"""

In [7]:
ground_truth = pd.read_csv("/content/jbs_scores_post.csv")
ground_truth.rename(columns={"u_id": "ID"}, inplace=True)
ground_truth["ID"] = ground_truth["ID"].astype(str)

In [8]:
data = pd.read_json("/content/all_participants_features.json")
data.head()

,1002,1102,1103,1105,1108,1115,1150,1151,1153,1154,...,1307,1308,1309,1310,1311,1312,1313,1314,1315,1316
Caffeine Intake,"{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.5, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...",...,"{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon..."
Evening Energy,"{'energy_evening_median': 24.0, 'energy_evenin...","{'energy_evening_median': 41.0, 'energy_evenin...","{'energy_evening_median': 68.0, 'energy_evenin...","{'energy_evening_median': 80.0, 'energy_evenin...","{'energy_evening_median': 28.0, 'energy_evenin...","{'energy_evening_median': 70.5, 'energy_evenin...","{'energy_evening_median': 30.0, 'energy_evenin...","{'energy_evening_median': 35.0, 'energy_evenin...","{'energy_evening_median': 43.0, 'energy_evenin...","{'energy_evening_median': 31.0, 'energy_evenin...",...,"{'energy_evening_median': 16.0, 'energy_evenin...","{'energy_evening_median': 36.0, 'energy_evenin...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 30.0, 'energy_evenin...","{'energy_evening_median': 10.0, 'energy_evenin...","{'energy_evening_median': 12.5, 'energy_evenin...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening..."
Heart Rate Variability,"{'RestingHeartRate_std': 3.201, 'hrentropy_std...","{'RestingHeartRate_std': 2.171, 'hrentropy_std...","{'RestingHeartRate_std': 1.956, 'hrentropy_std...","{'RestingHeartRate_std': 1.687, 'hrentropy_std...","{'RestingHeartRate_std': 2.103, 'hrentropy_std...","{'RestingHeartRate_std': 1.6520000000000001, '...","{'RestingHeartRate_std': 2.318, 'hrentropy_std...","{'RestingHeartRate_std': 1.324, 'hrentropy_std...","{'RestingHeartRate_std': 2.082, 'hrentropy_std...","{'RestingHeartRate_std': 1.974, 'hrentropy_std...",...,"{'RestingHeartRate_std': 2.659, 'hrentropy_std...","{'RestingHeartRate_std': 4.312, 'hrentropy_std...","{'RestingHeartRate_std': 3.65, 'hrentropy_std'...","{'RestingHeartRate_std': 3.229, 'hrentropy_std...","{'RestingHeartRate_std': 2.304, 'hrentropy_std...","{'RestingHeartRate_std': 1.867, 'hrentropy_std...","{'RestingHeartRate_std': 2.96, 'hrentropy_std'...","{'RestingHeartRate_std': 2.972, 'hrentropy_std...","{'RestingHeartRate_std': 3.002, 'hrentropy_std...","{'RestingHeartRate_std': 3.3, 'hrentropy_std':..."
Inactivity at Daytime,{'duration_entropy_non_step_std': 0.784},{'duration_entropy_non_step_std': 0.242},{'duration_entropy_non_step_std': 0.224},{'duration_entropy_non_step_std': 0.258},{'duration_entropy_non_step_std': 0.331},{'duration_entropy_non_step_std': 0.354},{'duration_entropy_non_step_std': 0.304},{'duration_entropy_non_step_std': 0.267},{'duration_entropy_non_step_std': 0.268},{'duration_entropy_non_step_std': 0.3460000000...,...,{'duration_entropy_non_step_std': 0.384},{'duration_entropy_non_step_std': 0.544},{'duration_entropy_non_step_std': 0.303},{'duration_entropy_non_step_std': 0.438},{'duration_entropy_non_step_std': 0.335},{'duration_entropy_non_step_std': 0.242},{'duration_entropy

In [15]:
import numpy as np

participant_ids = list(data.columns)
batch_size = 4
NUM_RUNS = 3
results = []

metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

print(f"\nProcessing {len(participant_ids)} participants in batches of {batch_size}...")

def clean_label(label):
    label = str(label).capitalize()
    if "High" in label: return "High"
    if "Low" in label: return "Low"
    return "Unknown"

for run in range(NUM_RUNS):
    print(f"\n{'='*40}")
    print(f"STARTING RUN {run + 1}/{NUM_RUNS}")
    print(f"{'='*40}")

    results = []

    print(f"Processing {len(participant_ids)} participants in batches of {batch_size}...")
    for i in tqdm(range(0, len(participant_ids), batch_size)):
        batch_ids = participant_ids[i:i + batch_size]
        batch_series = [data[pid] for pid in batch_ids]

        # --- STAGE 1: Summarization ---
        summary_prompts = [build_prompt(series) for series in batch_series]

        summary_inputs = tokenizer(
            summary_prompts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            summary_outputs = model.generate(
                **summary_inputs,
                max_new_tokens=250,
                temperature=0.3, # Temperature allows slight variations between runs
                do_sample=True
            )

        summaries = tokenizer.batch_decode(summary_outputs, skip_special_tokens=True)

        # --- STAGE 2: Classification ---
        class_prompts = [build_classification_prompt(summary) for summary in summaries]

        class_inputs = tokenizer(
            class_prompts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            class_outputs = model.generate(
                **class_inputs,
                max_new_tokens=3,
                temperature=0.0,
                do_sample=False
            )

        classifications = tokenizer.batch_decode(class_outputs, skip_special_tokens=True)

        # --- Save Batch Results ---
        for pid, summary, label in zip(batch_ids, summaries, classifications):
            results.append({
                "ID": pid,
                "llm_summary": summary,
                "llm_burnout_label": label.strip()
            })

    # Save CSV uniquely for each run
    df_results = pd.DataFrame(results)
    df_results.to_csv(f"t5_llm_predictions_run_{run + 1}.csv", index=False)

    # --- Run Evaluation ---
    df_results["ID"] = df_results["ID"].astype(str)

    df_eval = df_results.merge(ground_truth, on="ID", how="inner")
    df_eval["llm_burnout_label"] = df_eval["llm_burnout_label"].apply(clean_label)

    # Fix: Convert the numeric risk labels to strings so they match the predictions
    df_eval["risk"] = df_eval["risk"].map({1: "High", 0: "Low"})

    # Calculate Metrics for this specific run
    # Note: Using average='weighted' to safely handle potential 'Unknown' labels generated by the LLM
    acc = accuracy_score(df_eval["risk"], df_eval["llm_burnout_label"])
    prec = precision_score(df_eval["risk"], df_eval["llm_burnout_label"], average='weighted', zero_division=0)
    rec = recall_score(df_eval["risk"], df_eval["llm_burnout_label"], average='weighted', zero_division=0)
    f1 = f1_score(df_eval["risk"], df_eval["llm_burnout_label"], average='weighted', zero_division=0)

    # Store metrics
    metrics["accuracy"].append(acc)
    metrics["precision"].append(prec)
    metrics["recall"].append(rec)
    metrics["f1"].append(f1)

    print(f"\nRun {run + 1} Metrics:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("\nDetailed Report for this run:")
    print(classification_report(df_eval["risk"], df_eval["llm_burnout_label"], zero_division=0))

# --- AGGREGATED RESULTS ---
print(f"\n{'='*40}")
print(f"FINAL AGGREGATED RESULTS (OVER {NUM_RUNS} RUNS)")
print(f"{'='*40}")

for metric_name, values in metrics.items():
    avg = np.mean(values)
    std = np.std(values)
    print(f"{metric_name.capitalize()}: {avg:.4f} ± {std:.4f}")


Processing 44 participants in batches of 4...

STARTING RUN 1/3
Processing 44 participants in batches of 4...


100%|██████████| 11/11 [01:28<00:00,  8.03s/it]



Run 1 Metrics:
Accuracy:  0.3636
Precision: 0.6534
Recall:    0.3636
F1 Score:  0.4558

Detailed Report for this run:
              precision    recall  f1-score   support

        High       0.75      0.39      0.52        38
         Low       0.04      0.17      0.07         6

    accuracy                           0.36        44
   macro avg       0.40      0.28      0.29        44
weighted avg       0.65      0.36      0.46        44


STARTING RUN 2/3
Processing 44 participants in batches of 4...


100%|██████████| 11/11 [01:36<00:00,  8.81s/it]



Run 2 Metrics:
Accuracy:  0.5227
Precision: 0.7398
Recall:    0.5227
F1 Score:  0.5976

Detailed Report for this run:
              precision    recall  f1-score   support

        High       0.84      0.55      0.67        38
         Low       0.11      0.33      0.16         6

    accuracy                           0.52        44
   macro avg       0.47      0.44      0.41        44
weighted avg       0.74      0.52      0.60        44


STARTING RUN 3/3
Processing 44 participants in batches of 4...


100%|██████████| 11/11 [01:30<00:00,  8.22s/it]


Run 3 Metrics:
Accuracy:  0.5682
Precision: 0.8145
Recall:    0.5682
F1 Score:  0.6350

Detailed Report for this run:
              precision    recall  f1-score   support

        High       0.91      0.55      0.69        38
         Low       0.19      0.67      0.30         6

    accuracy                           0.57        44
   macro avg       0.55      0.61      0.49        44
weighted avg       0.81      0.57      0.64        44


FINAL AGGREGATED RESULTS (OVER 3 RUNS)
Accuracy: 0.4848 ± 0.0877
Precision: 0.7359 ± 0.0658
Recall: 0.4848 ± 0.0877
F1: 0.5628 ± 0.0772
